## Create Geographic Footprints from SSUSA

This notebook builds three spatial products from cleaned SSUSA camera locations:

1. **Camera-level footprints**: a 1 km buffer around each unique camera location  
2. **Array-level footprints**: merged 1 km buffers for all cameras within each `Camera_Trap_Array`  
3. **Combined sampled-area footprint**: one unioned polygon representing the full buffered area sampled by all cameras

These outputs are useful for downstream spatial comparisons with external biodiversity reference layers such as IUCN mammal ranges.

In [3]:
import os

import geopandas as gpd
import pandas as pd
from shapely.ops import unary_union

### Load data from shape files and SSUSA cleaned CSV

In [4]:
base_iucn_folder = "../../data/iucn"
IUCN_SHP = "MAMMALS_TERRESTRIAL_ONLY/MAMMALS_TERRESTRIAL_ONLY.shp"
IUCN_XML = "MAMMALS_TERRESTRIAL_ONLY/MAMMALS_TERRESTRIAL_ONLY.shp.xml"

# Local Natural Earth shapefile path (not directly used in this notebook)
WORLD_SHP = "../../data/ne_110m_admin_0_countries/ne_110m_admin_0_countries.shp"

# Cleaned SSUSA CSV
base_ssusa_path = "../../data/ssusa"
SSUSA_CSV = "cleaned_snapshot_usa_iucn.csv"

# Output folder for generated GeoJSON files
OUTPUT_PATH = "../../outputs"

# Buffer settings
BUFFER_KM = 1
buffer_m = BUFFER_KM * 1000  # convert km to meters

# Projected CRS used for buffering in meters
AEA = "EPSG:5070"

# Coordinate column names in the SSUSA CSV
LON_COL = "Longitude"
LAT_COL = "Latitude"

# Make sure the output folder exists before writing files
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [5]:
# Read cleaned SSUSA data
ssusa_file = os.path.join(base_ssusa_path, SSUSA_CSV)
ssusa_df = pd.read_csv(ssusa_file, low_memory=False)

# Preview the first few rows
ssusa_df.head()

,Year,Project,Camera_Trap_Array,Deployment_ID,Sequence_ID,Start_Time,End_Time,Class,Order,Family,...,Group_Size,Site_Name,Start_Date,End_Date,Survey_Nights,Latitude,Longitude,Habitat,Development_Level,Feature_Type
0,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s1,2019-08-31 06:50:00,2019-08-31 06:50:00,mammalia,carnivora,ursidae,...,1,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64.0,59.42643,-136.2225,forest,wild,water source
1,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s2,2019-08-31 14:15:00,2019-08-31 14:17:00,mammalia,carnivora,ursidae,...,1,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64.0,59.42643,-136.2225,forest,wild,water source
2,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s3,2019-08-31 18:22:00,2019-08-31 18:22:00,mammalia,carnivora,ursidae,...,1,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64.0,59.42643,-136.2225,forest,wild,water source
3,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s4,2019-08-31 20:58:00,2019-08-31 20:58:00,mammalia,carnivora,ursidae,...,1,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64.0,59.42643,-136.2225,forest,wild,water source
4,2019,Snapshot USA 2019,Crupi,AK_Forest_Chilkat_Preserve_1,d58722s4,2019-08-31 20:58:00,2019-08-31 20:58:00,mammalia,carnivora,ursidae,...,2,AK_Forest_Crupi_21_dep_01,2019-08-31,2019-11-03,64.0,59.42643,-136.2225,forest,wild,water source


### Create camera-level footprints

At the camera level, we want **one 1 km buffer per unique camera location**.

This section:
- keeps only valid longitude/latitude rows
- removes duplicate coordinates
- converts points to a GeoDataFrame
- reprojects to a meter-based CRS
- buffers each point by 1 km
- checks for invalid or duplicate geometries

In [6]:
print("Total SSUSA rows:", len(ssusa_df))

# Keep one row per unique camera location
camera_df = (
    ssusa_df[[LON_COL, LAT_COL]]
    .dropna(subset=[LON_COL, LAT_COL])
    .drop_duplicates(subset=[LON_COL, LAT_COL])
    .copy()
)

print("Unique camera locations:", len(camera_df))

Total SSUSA rows: 698887
Unique camera locations: 7303


In [7]:
# Convert longitude/latitude into point geometry in WGS84
camera_points = gpd.GeoDataFrame(
    camera_df,
    geometry=gpd.points_from_xy(camera_df[LON_COL], camera_df[LAT_COL]),
    crs="EPSG:4326"
)

# Reproject so the buffer distance is interpreted in meters
camera_points_aea = camera_points.to_crs(AEA)

# Create a 1 km buffer around each camera point
camera_footprints = camera_points_aea.copy()
camera_footprints["geometry"] = camera_footprints.geometry.buffer(buffer_m)

# Remove any null, empty, or invalid geometries
camera_footprints = camera_footprints[
    camera_footprints.geometry.notnull()
    & ~camera_footprints.geometry.is_empty
    & camera_footprints.geometry.is_valid
].copy()

# Basic diagnostics
print(f"Total unique camera points:   {len(camera_points_aea)}")
print(f"Total camera footprints kept: {len(camera_footprints)}")
print(f"Invalid footprints:           {(~camera_footprints.geometry.is_valid).sum()}")
print(f"Empty footprints:             {(camera_footprints.geometry.is_empty).sum()}")
print(
    "Duplicate footprints:",
    camera_footprints.duplicated(subset=["geometry"]).sum()
)

Total unique camera points:   7303
Total camera footprints kept: 7303
Invalid footprints:           0
Empty footprints:             0
Duplicate footprints: 0


In [8]:
# -------------------------------------------------------------------
# Save camera-level footprints
# -------------------------------------------------------------------
camera_footprints.to_file(
    os.path.join(OUTPUT_PATH, f"ssusa_camera_footprints_{BUFFER_KM}km.geojson"),
    driver="GeoJSON"
)

### Create array-level footprints

At the array level, we want to preserve the camera-to-array relationship first, then merge all camera buffers within each Camera_Trap_Array.

This section:
- keeps Camera_Trap_Array with coordinates
- creates camera points
- buffers each camera by 1 km
- dissolves all buffers within each array
- repairs invalid geometries after dissolve

In [9]:
# Keep array ID together with each distinct camera location
array_camera_df = (
    ssusa_df[["Camera_Trap_Array", LON_COL, LAT_COL]]
    .dropna(subset=[LON_COL, LAT_COL])
    .drop_duplicates()
    .copy()
)

# Convert array-linked camera locations into point geometry
array_camera_points = gpd.GeoDataFrame(
    array_camera_df,
    geometry=gpd.points_from_xy(array_camera_df[LON_COL], array_camera_df[LAT_COL]),
    crs="EPSG:4326"
)

# Reproject before buffering
array_camera_points_aea = array_camera_points.to_crs(AEA)

# Create a 1 km buffer around each array-linked camera point
array_camera_buffers = array_camera_points_aea.copy()
array_camera_buffers["geometry"] = array_camera_buffers.geometry.buffer(buffer_m)

# Remove null, empty, or invalid geometries
array_camera_buffers = array_camera_buffers[
    array_camera_buffers.geometry.notnull()
    & ~array_camera_buffers.geometry.is_empty
    & array_camera_buffers.geometry.is_valid
].copy()

print("Unique array-linked camera locations:", len(array_camera_buffers))

Unique array-linked camera locations: 7306


In [10]:
# check if same camera array was shared by more than one camera location
coord_array_counts = (
    ssusa_df[["Camera_Trap_Array", LON_COL, LAT_COL]]
    .dropna(subset=[LON_COL, LAT_COL])
    .drop_duplicates()
    .groupby([LON_COL, LAT_COL])["Camera_Trap_Array"]
    .nunique()
    .reset_index(name="n_arrays")
)

shared_coords = coord_array_counts[coord_array_counts["n_arrays"] > 1].copy()

print("Number of coordinate pairs used in multiple arrays:", len(shared_coords))
shared_coords.head(10)

Number of coordinate pairs used in multiple arrays: 3


,Longitude,Latitude,n_arrays
6104,-75.9000,35.80000,2
6105,-75.9000,35.90000,2
6949,-71.7209,41.55074,2


In [11]:
# Merge all camera buffers within each array into one footprint
array_footprints = (
    array_camera_buffers
    .dissolve(by="Camera_Trap_Array")
    .reset_index()
)

# Repair topology if any merged geometries become invalid
array_footprints["geometry"] = array_footprints.geometry.make_valid()

print("Total array footprints:", len(array_footprints))


/Users/neelima/miniconda3/lib/python3.12/site-packages/shapely/set_operations.py:553: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


Total array footprints: 261


In [13]:
# Save array-level footprints
array_footprints.to_file(
    os.path.join(OUTPUT_PATH, f"ssusa_array_footprints_{BUFFER_KM}km.geojson"),
    driver="GeoJSON"
)

### Create one combined sampled-area footprint

This final footprint merges **all camera-level buffers** into a single geometry representing the full buffered area sampled by SSUSA.

This will be used for:
- clip large external datasets such as IUCN ranges
- reduce the spatial extent of later analyses
- summarize the total sampled area as one polygon

In [15]:
# Merge all camera-level footprints into one combined geometry
merged_geom = unary_union(camera_footprints.geometry)

# Store the merged geometry in a GeoDataFrame so CRS is preserved
sampled_area = gpd.GeoDataFrame(
    geometry=[merged_geom],
    crs=camera_footprints.crs
)

# Repair geometry in case the union introduces topology issues
sampled_area["geometry"] = sampled_area.geometry.make_valid()

# Quick validation checks
print("Merged geometry type:", sampled_area.geometry.iloc[0].geom_type)
print("Merged geometry valid:", sampled_area.geometry.is_valid.iloc[0])

Merged geometry type: MultiPolygon
Merged geometry valid: True


/Users/neelima/miniconda3/lib/python3.12/site-packages/shapely/set_operations.py:553: RuntimeWarning: invalid value encountered in unary_union
  return lib.unary_union(collections, **kwargs)


In [16]:
# Save combined sampled-area footprint

# Write the single merged SSUSA sampled-area polygon to GeoJSON.
sampled_area.to_file(
    os.path.join(
        OUTPUT_PATH,
        f"ssusa_all_cameras_sampled_footprint_{BUFFER_KM}km.geojson"
    ),
    driver="GeoJSON"
)